# Medir el efecto de la publicidad: la medición

Cuaderno de lectura de la medición `uplift-criteo/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos), sobre el *Criteo Uplift Prediction Dataset* 2.1 (CC BY-NC-SA 4.0). Lee los agregados que escribe `generar.py`; el fichero original, de unos 14 millones de filas, no se redistribuye y no hace falta para seguir este cuaderno. Las cifras describen el fichero publicado, que Criteo submuestreó de forma no uniforme antes de publicarlo: sirven para comparar métodos, no para reconstruir el efecto de las campañas originales.

Artículo: [Medir el efecto de la publicidad](https://manpla.net/posts/medir-efecto-publicidad/).

In [ ]:
import json, urllib.request
from pathlib import Path
import pandas as pd

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/uplift-criteo/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    return pd.read_csv(p if p.exists() else RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre) as r:
        return r.read().decode("utf-8")

R = json.loads(texto("resumen.json"))
print("filas:", R["filas"], "· tratamiento:", R["n_tratamiento"], "· control:", R["n_control"])
print("tasa de visita:", round(R["tasa_visita"], 6), "· conversión:", round(R["tasa_conversion"], 6))

## Los tres grupos: control, tratados sin anuncio y tratados expuestos

In [ ]:
g = leer("grupos.csv")
print(g.round(4).to_string(index=False))

## Equilibrio: diferencia estandarizada de cada atributo

In [ ]:
b = leer("balance.csv")
print(b[["atributo", "dem_tratamiento_control", "dem_expuestos_no_expuestos"]].round(3).to_string(index=False))

## Contrastes sobre el fichero completo

In [ ]:
c = leer("contrastes.csv")
print(c[["resultado", "metodo", "estimacion", "ic_inf", "ic_sup", "estadistico"]].round(5).to_string(index=False))

## Tamaño de muestra al 80 % y al 90 % de potencia (escenarios del fichero)

In [ ]:
t = leer("tamano_muestra.csv")
print(t.round(4).to_string(index=False))

## Siete modelos de uplift: área Qini y calibración

In [ ]:
m = leer("modelos.csv")
print(m[["modelo", "area_qini", "area_ic_inf", "area_ic_sup", "pendiente_calibracion"]].round(3).sort_values("area_qini", ascending=False, kind="stable").to_string(index=False))

## Segmentos: dónde se concentra el contraste incremental estimado

In [ ]:
s = leer("segmentos.csv")
print(s[["espacio", "grupo", "n", "cuota", "uplift"]].round(4).to_string(index=False))

## La consulta central en SQL con DuckDB

In [ ]:
print(texto("consulta_central.sql"))
print(leer("consulta_central.csv").round(5).T.to_string(header=False))